[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/14_kv_cache.ipynb)

# 🔴 Hard: KV Cache Attention

Implement **multi-head attention with KV caching** for efficient autoregressive generation.

During LLM inference, recomputing all key/value projections at every step is wasteful.
A **KV cache** stores previously computed K and V tensors so only the new token(s) need projection.

### Signature
```python
class KVCacheAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x: torch.Tensor, cache=None) -> tuple[torch.Tensor, tuple]:
        # x: (B, S_new, D) — new tokens
        # cache: None or (K_past, V_past) each (B, num_heads, S_past, d_k)
        # Returns: (output, (K_all, V_all))
```

### Requirements
- Inherit from `nn.Module`
- `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`: `nn.Linear` projections
- When `cache=None` (prefill): apply **causal mask**, return all K/V as cache
- When `cache` provided (decode): concat new K/V with cached, no causal mask needed for single-token decode
- Incremental decode must produce **identical** results to full forward pass

### Key Idea
```
Prefill:  [t0 t1 t2 t3] → full causal attention → cache = (K_{0:3}, V_{0:3})
Decode:   [t4]           → Q=t4, K/V=cache+t4  → cache = (K_{0:4}, V_{0:4})
Decode:   [t5]           → Q=t5, K/V=cache+t5  → cache = (K_{0:5}, V_{0:5})
```

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.7 MB/s eta 0:00:00


In [36]:
import torch
import torch.nn as nn
import math
import torch.nn.functional as F

In [48]:
# ✏️ YOUR IMPLEMENTATION HERE

class KVCacheAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads

        # Initialize W_q, W_k, W_v, W_o
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)


    def forward(self, x, cache=None):
        # 1. Project Q, K, V from x

        # x: (B, S_new, D)
        q = self.W_q(x) # q: (B, S_new, D)
        k = self.W_k(x) # k: (B, S_new, D)
        v = self.W_v(x) # v: (B, S_new, D)

        B = q.shape[0]
        S = q.shape[1]
        D = q.shape[2]
        assert (D == self.d_model)
        d_k = self.d_model // self.num_heads

        # 2. Reshape to multi-head: (B, num_heads, S, d_k)

        q = torch.reshape(q, (B, S, self.num_heads, d_k))
        k = torch.reshape(k, (B, S, self.num_heads, d_k))
        v = torch.reshape(v, (B, S, self.num_heads, d_k))

        q = torch.transpose(q, 1, 2)
        k = torch.transpose(k, 1, 2)
        v = torch.transpose(v, 1, 2)

        # 3. If cache exists, concat new K/V with cached K/V
        if cache is not None:
            cache_k, cache_v = cache
            cache_k = torch.cat((cache_k, k), dim=2)
            cache_v = torch.cat((cache_v, v), dim=2)

        else:
            cache_k = k
            cache_v = v

        # 4. Compute attention (causal mask needed during prefill)

        if cache is None: # prefill
            #print(q.shape, cache_k.shape)

            score = torch.einsum("bhsk,bhtk->bhst", q, k)
            score = torch.div(score, math.sqrt(d_k))

            S_total = cache_k.size(2)
            S_past = S_total - S
            if S > 1:
                q_pos = torch.arange(S_past, S_total).unsqueeze(1)  # (S, 1)
                k_pos = torch.arange(S_total).unsqueeze(0)          # (1, S_total)
                score = score.masked_fill(k_pos > q_pos, float('-inf'))

            score = F.softmax(score, dim=-1)
            score = torch.einsum("bhst, bhtk->bhsk", score, v)

        else:
            #print(q.shape, cache_k.shape)
            score = torch.einsum("bhsk,bhtk->bhst", q, cache_k)
            score = torch.div(score, math.sqrt(d_k))
            score = F.softmax(score, dim=-1)
            score = torch.einsum("bhst, bhtk->bhsk", score, cache_v)

        score = score.transpose(1, 2)
        score = torch.reshape(score, (B, S, self.d_model))
        output = self.W_o(score)


        # 5. Return (output, (K_all, V_all))
        return (output, (cache_k, cache_v))


In [50]:
# 🧪 Debug
torch.manual_seed(0)
attn = KVCacheAttention(d_model=64, num_heads=4)
x = torch.randn(1, 6, 64)

# Full forward
full_out, _ = attn(x)
print("Full output shape:", full_out.shape)  # (1, 6, 64)
#print(full_out)

# Incremental: prefill 4, decode 1, decode 1
out1, cache = attn(x[:, :4])
print("Cache K shape:", cache[0].shape)  # (1, 4, 4, 16)
#print(out1)
out2, cache = attn(x[:, 4:5], cache=cache)
#print(out2)
out3, cache = attn(x[:, 5:6], cache=cache)
#print(out3)
inc_out = torch.cat([out1, out2, out3], dim=1)
print("Match:", torch.allclose(full_out, inc_out, atol=1e-5))

Full output shape: torch.Size([1, 6, 64])
Cache K shape: torch.Size([1, 4, 4, 16])
Match: True


In [51]:
# ✅ SUBMIT
from torch_judge import check
check('kv_cache')


🧪 Testing: KV Cache Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (no cache) (9.0ms)
  ✅ [2/5] Cache structure (5.1ms)
  ✅ [3/5] Decode step appends to cache (5.8ms)
  ✅ [4/5] Incremental decode matches full forward (6.7ms)
  ✅ [5/5] Gradient flow (6.4ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (33.0ms total)
  Progress saved. Run status() to see your dashboard.

